# GPU ODE Benchmark Suite

Modular problem library + GPU divergence-reduction strategies.

## Problem families
| ID | Name | Stiffness | State dim |
|----|------|-----------|----------|
| NS | Nonstiff oscillator | None (ω∈[1,4]) | d |
| MS | Mildly stiff decay | λ∈[1,10] | d |
| SS | Strongly stiff linear | λ∈[100,2000] | d |
| MX | Mixed stiffness (fast+slow modes) | κ∈[1,1000] | 2d |
| VZ | Van der Pol (scalar, μ∈[1,50]) | nonlinear stiff | 2 |
| RB | Robertson (3-species) | extreme stiff | 3 |
| LD | Large-dim stiff decay | λ∈[5,500] | 64 |

## GPU balancing strategies
| Strategy | Description |
|----------|-------------|
| `baseline` | Mask inactive (current approach) |
| `compaction` | Active-set compaction each iteration |
| `buckets` | Partition by stiffness regime at start |
| `rebatch` | Periodic compaction every K steps |
| `split_queue` | Separate explicit/implicit queues |

## Instrumentation
Active fraction, step-size spread (CoV), rejection rate, Newton iterations/step, compaction overhead.

In [18]:
#%pip install -q -U "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
import os, time, math, pickle
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

jax.config.update('jax_enable_x64', True)

# Mount Google Drive (Colab only — harmless no-op elsewhere)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not in Colab — Drive checkpointing disabled')

print('JAX version   :', jax.__version__)
print('Devices       :', jax.devices())
print('Default backend:', jax.default_backend())
print('CPU cores     :', os.cpu_count())


JAX version   : 0.10.0
Devices       : [CpuDevice(id=0)]
Default backend: cpu
CPU cores     : 2


In [ ]:
# ---------------------------------------------------------------------------
# Drive checkpoint helpers — atomic write via tmp-file + os.replace
# ---------------------------------------------------------------------------
DRIVE_DIR  = '/content/drive/MyDrive/ODE_bench_results'
CKPT_MAIN  = os.path.join(DRIVE_DIR, 'all_results.pkl')
CKPT_SCALE = os.path.join(DRIVE_DIR, 'scale_records.pkl')
CKPT_LD    = os.path.join(DRIVE_DIR, 'ld_records.pkl')

def _ensure_drive_dir():
    if IN_COLAB:
        os.makedirs(DRIVE_DIR, exist_ok=True)

def save_to_drive(obj, path):
    """Atomically pickle obj to path on Drive (no-op outside Colab)."""
    if not IN_COLAB:
        return
    _ensure_drive_dir()
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, path)
    print(f'  [ckpt] saved → {os.path.basename(path)}')

def load_from_drive(path):
    """Load pickle from Drive; returns None if not in Colab or file absent."""
    if not IN_COLAB or not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)

print('Checkpoint helpers defined')
print(f'Drive dir : {DRIVE_DIR}')


## 1. Problem Interface

In [19]:
@dataclass
class ODEProblem:
    """Common interface for all ODE test problems."""
    name: str
    state_dim: int                              # d per trajectory
    t_span: Tuple[float, float]
    rhs: Callable                               # (t, y, params) -> dy/dt  [JAX-compatible]
    jac: Optional[Callable]                     # (t, y, params) -> d×d Jacobian
    ic_generator: Callable                      # (rng, n_traj) -> (n,d) array
    param_sampler: Callable                     # (rng, n_traj) -> params array
    ref_solution: Optional[Callable]            # (t_end, y0, params) -> y(t_end)
    # stiffness metadata
    stiffness_label: str = 'unknown'            # 'nonstiff'/'mild'/'strong'/'mixed'/'extreme'
    expected_stiffness_ratio: float = 1.0       # rough |λ_max/λ_min|
    # recommended solver hint
    solver_hint: str = 'explicit'               # 'explicit' | 'implicit'


def _check_rng(rng):
    if isinstance(rng, np.random.Generator):
        return rng
    return np.random.default_rng(rng)

## 2. Problem Library

In [20]:
# ---------------------------------------------------------------------------
# NS: Nonstiff harmonic oscillator  y'' + ω²y = 0
#     State: [y, y'], params: [ω] per mode, d modes decoupled
#     Exact: y_i(t) = a_i cos(ω_i t) + b_i sin(ω_i t)
# ---------------------------------------------------------------------------
def _ns_rhs(t, y, params):
    # y shape (2d,): [y0,y0',y1,y1',...]
    d = params.shape[0]
    yp = y.reshape(d, 2)
    dy = jnp.stack([
        yp[:, 1],
        -params**2 * yp[:, 0]
    ], axis=1).reshape(-1)
    return dy

def _ns_ref(t_end, y0, params):
    d = params.shape[0]
    y0r = y0.reshape(d, 2)
    a, b = y0r[:, 0], y0r[:, 1] / params
    ct, st = np.cos(params * t_end), np.sin(params * t_end)
    y_end = np.stack([a*ct + b*st, (-a*params*st + b*params*ct)], axis=1)
    return y_end.reshape(-1)

def make_nonstiff_problem(d=4, t_end=10.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, 2*d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(1.0, 4.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_ns_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='NS', state_dim=2*d, t_span=(0., t_end),
        rhs=_ns_rhs, jac=None,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='nonstiff', expected_stiffness_ratio=1.0,
        solver_hint='explicit'
    )

print('NS problem defined')

NS problem defined


In [21]:
# ---------------------------------------------------------------------------
# MS: Mildly stiff decoupled linear decay
#     y_i' = -λ_i (y_i - cos t) - sin t,  λ_i ∈ [1, 10]
# ---------------------------------------------------------------------------
def _stiff_lin_rhs(t, y, params):
    return -params * (y - jnp.cos(t)) - jnp.sin(t)

def _stiff_lin_jac(t, y, params):
    return -jnp.diag(params)

def _stiff_lin_ref(t_end, y0, params):
    return (y0 - 1.0) * np.exp(-params * t_end) + np.cos(t_end)

def make_mildly_stiff_problem(d=4, t_end=6.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(1.0, 10.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_stiff_lin_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='MS', state_dim=d, t_span=(0., t_end),
        rhs=_stiff_lin_rhs, jac=_stiff_lin_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='mild', expected_stiffness_ratio=10.0,
        solver_hint='explicit'
    )


# ---------------------------------------------------------------------------
# SS: Strongly stiff decoupled linear decay
#     Same RHS, λ_i ∈ [100, 2000]
# ---------------------------------------------------------------------------
def make_strongly_stiff_problem(d=4, t_end=1.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(100.0, 2000.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_stiff_lin_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='SS', state_dim=d, t_span=(0., t_end),
        rhs=_stiff_lin_rhs, jac=_stiff_lin_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='strong', expected_stiffness_ratio=2000.0,
        solver_hint='implicit'
    )

print('MS, SS problems defined')

MS, SS problems defined


In [22]:
# ---------------------------------------------------------------------------
# MX: Mixed-stiffness — fast modes (λ_fast) coupled to slow modes (λ_slow)
#     y_fast_i' = -λ_fast_i * y_fast_i + ε * y_slow_i
#     y_slow_i' = -λ_slow_i * y_slow_i
#     params: [λ_fast (d,), λ_slow (d,)] concatenated → (2d,)
# ---------------------------------------------------------------------------
def _mx_rhs(t, y, params):
    d = params.shape[0] // 2
    lf, ls = params[:d], params[d:]
    yf, ys = y[:d], y[d:]
    eps = 0.01
    dyf = -lf * yf + eps * ys
    dys = -ls * ys
    return jnp.concatenate([dyf, dys])

def _mx_jac(t, y, params):
    d = params.shape[0] // 2
    lf, ls = params[:d], params[d:]
    eps = 0.01
    J = jnp.zeros((2*d, 2*d))
    J = J.at[:d, :d].set(-jnp.diag(lf))
    J = J.at[:d, d:].set(eps * jnp.eye(d))
    J = J.at[d:, d:].set(-jnp.diag(ls))
    return J

def make_mixed_stiffness_problem(d=4, t_end=2.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(0.5, 2.0, size=(n, 2*d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        lf = rng.uniform(500.0, 1000.0, size=(n, d))
        ls = rng.uniform(0.5,   5.0,    size=(n, d))
        return np.concatenate([lf, ls], axis=1).astype(np.float64)
    return ODEProblem(
        name='MX', state_dim=2*d, t_span=(0., t_end),
        rhs=_mx_rhs, jac=_mx_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='mixed', expected_stiffness_ratio=1000.0,
        solver_hint='implicit'
    )

print('MX problem defined')

MX problem defined


In [23]:
# ---------------------------------------------------------------------------
# VZ: Van der Pol oscillator (stiffness increases with μ)
#     y1' = y2
#     y2' = μ(1 - y1²)y2 - y1
#     params: [μ],  μ ∈ [1, 50]
# ---------------------------------------------------------------------------
def _vz_rhs(t, y, params):
    mu = params[0]
    return jnp.array([y[1], mu * (1. - y[0]**2) * y[1] - y[0]])

def _vz_jac(t, y, params):
    mu = params[0]
    return jnp.array([
        [0.,                          1.],
        [-2.*mu*y[0]*y[1] - 1.,  mu*(1. - y[0]**2)]
    ])

def make_vanderpol_problem(t_end=10.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        y1 = rng.uniform(1.5, 2.5, size=(n, 1))
        y2 = rng.uniform(-1., 1., size=(n, 1))
        return np.concatenate([y1, y2], axis=1).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(1.0, 50.0, size=(n, 1)).astype(np.float64)
    return ODEProblem(
        name='VZ', state_dim=2, t_span=(0., t_end),
        rhs=_vz_rhs, jac=_vz_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='strong', expected_stiffness_ratio=50.0,
        solver_hint='implicit'
    )

print('VZ problem defined')

VZ problem defined


In [24]:
# ---------------------------------------------------------------------------
# RB: Robertson chemical kinetics (extremely stiff, 3 species)
#     y1' = -k1*y1 + k3*y2*y3
#     y2' =  k1*y1 - k2*y2^2 - k3*y2*y3
#     y3' =  k2*y2^2
#     params: [k1, k2, k3]  (nominal: 0.04, 3e7, 1e4; varied ±25%)
#     t_end=100 — avoids h_min stall; at 1e4 the trapezoidal error estimate
#     forces h→h_min near t=0 (BE cannot span the full 12-decade λ range).
# ---------------------------------------------------------------------------
def _rb_rhs(t, y, params):
    k1, k2, k3 = params[0], params[1], params[2]
    y1, y2, y3 = y[0], y[1], y[2]
    return jnp.array([
        -k1*y1 + k3*y2*y3,
         k1*y1 - k2*y2**2 - k3*y2*y3,
         k2*y2**2
    ])

def _rb_jac(t, y, params):
    k1, k2, k3 = params[0], params[1], params[2]
    y1, y2, y3 = y[0], y[1], y[2]
    return jnp.array([
        [-k1,           k3*y3,         k3*y2],
        [ k1, -2.*k2*y2-k3*y3,        -k3*y2],
        [0.,        2.*k2*y2,           0.  ]
    ])

def make_robertson_problem(t_end=100.0):
    def ic_gen(rng, n):
        return np.tile([1., 0., 0.], (n, 1)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        scale   = rng.uniform(0.75, 1.25, size=(n, 3))
        nominal = np.array([0.04, 3e7, 1e4])
        return (scale * nominal).astype(np.float64)
    return ODEProblem(
        name='RB', state_dim=3, t_span=(0., t_end),
        rhs=_rb_rhs, jac=_rb_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='extreme', expected_stiffness_ratio=1e10,
        solver_hint='implicit'
    )

print('RB problem defined')


RB problem defined


In [25]:
# ---------------------------------------------------------------------------
# LD: Large-dimensional stiff linear decay  (d=64)
#     Tests GPU register pressure and batched matrix-solve scaling.
#     RHS/Jac reuse _stiff_lin_rhs / _stiff_lin_jac from the MS/SS cell.
# ---------------------------------------------------------------------------
def make_large_dim_problem(d=64, t_end=1.0):
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        return rng.normal(size=(n, d)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(5.0, 500.0, size=(n, d)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        return np.array([_stiff_lin_ref(t_end, y0_batch[i], params_batch[i])
                         for i in range(len(y0_batch))])
    return ODEProblem(
        name='LD', state_dim=d, t_span=(0., t_end),
        rhs=_stiff_lin_rhs, jac=_stiff_lin_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='strong', expected_stiffness_ratio=100.0,
        solver_hint='implicit'
    )

print('LD problem defined')


LD problem defined
  NS: state_dim=8, stiffness=nonstiff, solver_hint=explicit
  MS: state_dim=4, stiffness=mild, solver_hint=explicit
  SS: state_dim=4, stiffness=strong, solver_hint=implicit
  MX: state_dim=8, stiffness=mixed, solver_hint=implicit
  VZ: state_dim=2, stiffness=strong, solver_hint=implicit
  RB: state_dim=3, stiffness=extreme, solver_hint=implicit
  LD: state_dim=64, stiffness=strong, solver_hint=implicit


In [ ]:
# ---------------------------------------------------------------------------
# LZ: Lorenz attractor — chaotic, mildly stiff, extreme step-size divergence
#     x' = σ(y - x),  y' = x(ρ-z) - y,  z' = xy - βz
#     params: [σ, ρ, β]   σ=10 fixed, ρ ∈ [20,35], β=8/3 fixed
#     Batch over ρ and IC noise: chaotic sensitivity → rapid step-size spread
#     across the batch.  Good stress test for COMPACTION and BUCKETS.
# ---------------------------------------------------------------------------
def _lz_rhs(t, u, params):
    sig, rho, beta = params[0], params[1], params[2]
    return jnp.array([
        sig  * (u[1] - u[0]),
        u[0] * (rho - u[2]) - u[1],
        u[0] * u[1] - beta * u[2],
    ])

def _lz_jac(t, u, params):
    sig, rho, beta = params[0], params[1], params[2]
    return jnp.array([
        [-sig,          sig,       0.   ],
        [rho - u[2],   -1.,       -u[0] ],
        [u[1],          u[0],     -beta ],
    ])

def make_lorenz_problem(t_end=10.0):
    def ic_gen(rng, n):
        rng  = _check_rng(rng)
        base = np.array([1.0, 0.0, 0.0])
        return (base + rng.normal(scale=0.5, size=(n, 3))).astype(np.float64)
    def param_sampler(rng, n):
        rng  = _check_rng(rng)
        sig  = np.full(n, 10.0)
        rho  = rng.uniform(20.0, 35.0, size=n)
        beta = np.full(n, 8.0 / 3.0)
        return np.stack([sig, rho, beta], axis=1).astype(np.float64)
    return ODEProblem(
        name='LZ', state_dim=3, t_span=(0., t_end),
        rhs=_lz_rhs, jac=_lz_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=None,
        stiffness_label='mild', expected_stiffness_ratio=35.0,
        solver_hint='explicit',
    )

print('LZ (Lorenz attractor) problem defined')


In [ ]:
# ---------------------------------------------------------------------------
# HE: 1D Heat equation — method-of-lines  (parabolic PDE → stiff ODE)
#     u_t = α u_xx,  x ∈ (0,1),  u(0,t) = u(1,t) = 0
#     Spatial:  N=20 interior points,  Δx = 1/(N+1)
#     Scheme:   second-order finite difference → tridiagonal Jacobian
#     IC:       u(x,0) = A·sin(πx),  A ∈ [0.5, 2.0]
#     Exact ODE solution (single-mode IC = A·sin(πxᵢ)):
#       yᵢ(t) = yᵢ(0) · exp(λ₁ t)
#       λ₁ = −2(α/Δx²)·(1−cos(πΔx))   ← exact k=1 eigenvalue of tridiag matrix
#     Params:  [α],  α ∈ [0.1, 2.0]
#     Stiffness ratio ≈ 4(N+1)²/π² × α ≈ 178α   (strong for large α)
#     Key GPU test: batched 20×20 dense linear solve per Newton step.
# ---------------------------------------------------------------------------
_HE_N = 20
_HE_h = 1.0 / (_HE_N + 1)    # Δx = 1/21

def _he_rhs(t, y, params):
    alpha   = params[0]
    c       = alpha / (_HE_h * _HE_h)
    y_left  = jnp.concatenate([jnp.zeros(1), y[:-1]])   # ghost 0 at left BC
    y_right = jnp.concatenate([y[1:], jnp.zeros(1)])    # ghost 0 at right BC
    return c * (y_left - 2.0 * y + y_right)

def _he_jac(t, y, params):
    alpha = params[0]
    c     = alpha / (_HE_h * _HE_h)
    diag  = jnp.full(_HE_N, -2.0 * c)
    off   = jnp.full(_HE_N - 1, c)
    return jnp.diag(diag) + jnp.diag(off, 1) + jnp.diag(off, -1)

def make_heat_eq_problem(t_end=0.5):
    x_np = np.arange(1, _HE_N + 1) * _HE_h
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        A   = rng.uniform(0.5, 2.0, size=(n, 1))
        return (A * np.sin(np.pi * x_np)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(0.1, 2.0, size=(n, 1)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        # Exact discrete (ODE) solution: yᵢ(t) = yᵢ(0) · exp(λ₁·t)
        alpha  = params_batch[:, 0]
        c      = alpha / (_HE_h * _HE_h)
        lam1   = -2.0 * c * (1.0 - np.cos(np.pi * _HE_h))   # exact eigenvalue
        decay  = np.exp(lam1 * t_end)
        return y0_batch * decay[:, None]
    return ODEProblem(
        name='HE', state_dim=_HE_N, t_span=(0., t_end),
        rhs=_he_rhs, jac=_he_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='strong',
        expected_stiffness_ratio=178.0,
        solver_hint='implicit',
    )

print('HE (1D Heat Eq MoL, d=20, tridiag Jac) problem defined')


In [ ]:
# ---------------------------------------------------------------------------
# AD: 1D Advection equation — method-of-lines (hyperbolic PDE → nonstiff ODE)
#     u_t + c·u_x = 0,  x ∈ [0,1),  periodic BC
#     Spatial:  N=20 grid points, Δx = 1/N, first-order upwind
#     Scheme:   duᵢ/dt = −(c/Δx)·(uᵢ − uᵢ₋₁),   u₋₁ ≡ u_{N-1}
#     IC:       u(x,0) = A·sin(2πx),  A ∈ [0.5, 1.5]
#     Exact ODE solution via DFT of discrete eigenvalues:
#       λₖ = −(c/Δx)·(1−e^{−2πik/N}),   y(t) = IDFT(e^{λt}·DFT(y₀))
#     Params:  [c],  c ∈ [0.2, 2.0]  (positive wave speed → upwind stable)
#     Step-size spread: h_stable ∝ Δx/c → 10× spread for 10× range in c.
#     Key GPU test: explicit solver under maximum step-size divergence.
# ---------------------------------------------------------------------------
_AD_N  = 20
_AD_dx = 1.0 / _AD_N

def _ad_rhs(t, y, params):
    c      = params[0]
    y_left = jnp.roll(y, 1)           # periodic: y_{-1} = y_{N-1}
    return -(c / _AD_dx) * (y - y_left)

def _ad_jac(t, y, params):
    # J[i,i] = -coh,  J[i, (i-1)%N] = +coh  (upwind, periodic BC)
    # roll(I, -1, axis=1)[i,j] = 1  iff  j = (i-1) mod N
    c   = params[0]
    coh = c / _AD_dx
    return -coh * jnp.eye(_AD_N) + coh * jnp.roll(jnp.eye(_AD_N), -1, axis=1)

def make_advection_problem(t_end=1.0):
    x_np = np.arange(_AD_N) / _AD_N
    def ic_gen(rng, n):
        rng = _check_rng(rng)
        A   = rng.uniform(0.5, 1.5, size=(n, 1))
        return (A * np.sin(2.0 * np.pi * x_np)).astype(np.float64)
    def param_sampler(rng, n):
        rng = _check_rng(rng)
        return rng.uniform(0.2, 2.0, size=(n, 1)).astype(np.float64)
    def ref(t_end, y0_batch, params_batch):
        # Exact discrete-system solution via DFT propagation
        k_idx   = np.arange(_AD_N)
        results = np.empty_like(y0_batch)
        for i in range(len(y0_batch)):
            c_i  = params_batch[i, 0]
            lam  = -(c_i / _AD_dx) * (1.0 - np.exp(-2j * np.pi * k_idx / _AD_N))
            y0_k = np.fft.fft(y0_batch[i])
            results[i] = np.real(np.fft.ifft(y0_k * np.exp(lam * t_end)))
        return results
    return ODEProblem(
        name='AD', state_dim=_AD_N, t_span=(0., t_end),
        rhs=_ad_rhs, jac=_ad_jac,
        ic_generator=ic_gen, param_sampler=param_sampler,
        ref_solution=ref,
        stiffness_label='nonstiff',
        expected_stiffness_ratio=10.0,
        solver_hint='explicit',
    )

print('AD (1D Advection MoL, d=20, periodic BC) problem defined')


In [ ]:
# ---------------------------------------------------------------------------
# Problem registry — all 10 problems
# ---------------------------------------------------------------------------
PROBLEMS = {
    'NS': make_nonstiff_problem(d=4,     t_end=10.0),   # nonstiff harmonic oscillator
    'MS': make_mildly_stiff_problem(d=4, t_end=6.0),    # mild stiffness  λ∈[1,10]
    'SS': make_strongly_stiff_problem(d=4, t_end=1.0),  # strong stiffness λ∈[100,2000]
    'MX': make_mixed_stiffness_problem(d=4, t_end=2.0), # fast+slow coupled modes
    'VZ': make_vanderpol_problem(t_end=10.0),            # Van der Pol μ∈[1,50]
    'RB': make_robertson_problem(t_end=100.0),           # Robertson kinetics (extreme)
    'LD': make_large_dim_problem(d=64,   t_end=1.0),    # d=64 batched linalg stress
    'LZ': make_lorenz_problem(t_end=10.0),               # Lorenz chaos, h-spread test
    'HE': make_heat_eq_problem(t_end=0.5),               # PDE parabolic: 1D heat MoL
    'AD': make_advection_problem(t_end=1.0),             # PDE hyperbolic: 1D advection MoL
}

print(f'{"ID":>4} {"dim":>5} {"stiffness":>10} {"hint":>10} {"t_end":>8}  has_ref')
print('-' * 52)
for k, p in PROBLEMS.items():
    has_ref = p.ref_solution is not None
    print(f'{k:>4} {p.state_dim:>5} {p.stiffness_label:>10} {p.solver_hint:>10} '
          f'{p.t_span[1]:>8.3g}  {"yes" if has_ref else "no"}')


## 3. Core JAX Solvers (problem-agnostic)

In [26]:
# ---------------------------------------------------------------------------
# Instrumentation dataclass — accumulated during a solver run
# ---------------------------------------------------------------------------
from dataclasses import dataclass as _dc, field as _f

@_dc
class SolverStats:
    n_steps_total: int = 0           # total iterations (including inactive)
    active_fracs: list = _f(default_factory=list)   # per-step active fraction
    h_covs: list = _f(default_factory=list)         # per-step CoV of h across active
    n_accept: np.ndarray = None
    n_reject: np.ndarray = None
    n_newton: np.ndarray = None
    wall_time: float = 0.0
    compaction_overhead: float = 0.0

    def summary(self):
        af = np.array(self.active_fracs)
        hc = np.array(self.h_covs)
        rej_rate = (self.n_reject.sum() /
                    max(self.n_accept.sum() + self.n_reject.sum(), 1))
        d = {
            'wall_time': self.wall_time,
            'n_steps_total': self.n_steps_total,
            'mean_active_frac': float(af.mean()) if len(af) else 0.,
            'min_active_frac': float(af.min())  if len(af) else 0.,
            'mean_h_cov': float(hc.mean())      if len(hc) else 0.,
            'max_h_cov': float(hc.max())        if len(hc) else 0.,
            'rej_rate': float(rej_rate),
            'mean_accept': float(self.n_accept.mean()),
            'mean_newton': float(self.n_newton.mean()) if self.n_newton is not None else 0.,
            'compaction_overhead': self.compaction_overhead,
        }
        return d

    def print_summary(self, label=''):
        s = self.summary()
        print(f'{label}')
        print(f'  wall={s["wall_time"]:.3f}s  steps={s["n_steps_total"]}  '
              f'active={s["mean_active_frac"]:.2%}(min {s["min_active_frac"]:.2%})')
        print(f'  h_cov={s["mean_h_cov"]:.3f}(max {s["max_h_cov"]:.3f})  '
              f'rej_rate={s["rej_rate"]:.2%}  '
              f'mean_accept={s["mean_accept"]:.1f}  '
              f'mean_newton={s["mean_newton"]:.2f}  '
              f'compact_overhead={s["compaction_overhead"]:.3f}s')

print('SolverStats defined')

SolverStats defined


In [27]:
# ---------------------------------------------------------------------------
# Explicit RK45 (Dormand-Prince) — problem-agnostic via rhs callable
# ---------------------------------------------------------------------------
def _rk45_step(rhs, t, y, h, params):
    """Single DP5(4) step for one trajectory."""
    k1 = rhs(t,          y,                                                  params)
    k2 = rhs(t+h/5,      y+h*(1/5*k1),                                       params)
    k3 = rhs(t+3*h/10,   y+h*(3/40*k1+9/40*k2),                              params)
    k4 = rhs(t+4*h/5,    y+h*(44/45*k1-56/15*k2+32/9*k3),                    params)
    k5 = rhs(t+8*h/9,    y+h*(19372/6561*k1-25360/2187*k2
                               +64448/6561*k3-212/729*k4),                    params)
    k6 = rhs(t+h,        y+h*(9017/3168*k1-355/33*k2+46732/5247*k3
                               +49/176*k4-5103/18656*k5),                     params)
    y5 = y+h*(35/384*k1+500/1113*k3+125/192*k4-2187/6784*k5+11/84*k6)
    k7 = rhs(t+h, y5, params)
    y4 = y+h*(5179/57600*k1+7571/16695*k3+393/640*k4
              -92097/339200*k5+187/2100*k6+1/40*k7)
    return y5, y5-y4

def _err_norm(y_old, y_new, err, rtol, atol):
    sc = atol + rtol * jnp.maximum(jnp.abs(y_old), jnp.abs(y_new))
    return jnp.sqrt(jnp.mean((err/sc)**2, axis=-1))

print('RK45 step function defined')

RK45 step function defined


In [28]:
# ---------------------------------------------------------------------------
# Implicit BE (BDF1) Newton solver — problem-agnostic
# ---------------------------------------------------------------------------
def _newton_be(rhs, jac, y_prev, t_next, h, params, tol=1e-10, max_iter=20):
    """Batched backward-Euler Newton. Returns (y, converged, n_iters)."""
    B, d  = y_prev.shape
    eye   = jnp.eye(d)
    y     = y_prev.copy()
    conv  = jnp.zeros((B,), dtype=bool)
    iters = jnp.zeros((B,), jnp.int32)
    for k in range(max_iter):
        f   = jax.vmap(rhs, in_axes=(0, 0, 0))(t_next, y, params)
        res = y - y_prev - h[:, None] * f
        rn  = jnp.max(jnp.abs(res), axis=1)
        jc  = (~conv) & (rn < tol)
        iters = jnp.where(jc, k+1, iters)
        conv  = conv | jc
        if bool(jnp.all(conv)): break
        Jf  = jax.vmap(jac, in_axes=(0, 0, 0))(t_next, y, params)
        A   = eye[None] - h[:, None, None] * Jf
        dy  = jnp.linalg.solve(A, -res[..., None]).squeeze(-1)
        dy  = jnp.where(conv[:, None], 0., dy)
        y   = y + dy
        dn  = jnp.max(jnp.abs(dy), axis=1)
        jc2 = (~conv) & (dn < tol)
        iters = jnp.where(jc2, k+2, iters)
        conv  = conv | jc2
    iters = jnp.where((iters == 0) & conv, 1, iters)
    iters = jnp.where(~conv, max_iter, iters)
    return y, conv, iters

print('Newton BE solver defined')

Newton BE solver defined


## 4. GPU Balancing Strategies

In [29]:
# ---------------------------------------------------------------------------
# Strategy 1: BASELINE — masked batch (existing approach, no reordering)
# ---------------------------------------------------------------------------
def solve_baseline_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    t0, tf = problem.t_span
    rhs = problem.rhs
    use_implicit = (problem.solver_hint == 'implicit') and (problem.jac is not None)

    y      = jnp.array(y0_batch, dtype=jnp.float64)
    params = jnp.array(params_batch, dtype=jnp.float64)
    B      = y.shape[0]
    t      = jnp.full((B,), t0)
    h      = jnp.full((B,), h_init)
    done   = jnp.zeros((B,), dtype=bool)
    n_acc  = jnp.zeros((B,), jnp.int32)
    n_rej  = jnp.zeros((B,), jnp.int32)
    n_nit  = jnp.zeros((B,), jnp.int32)

    stats = SolverStats()
    t_start = time.perf_counter()

    for step in range(max_iters):
        active = ~done
        n_active = int(jnp.sum(active))
        if n_active == 0:
            break

        # Instrumentation
        stats.active_fracs.append(n_active / B)
        h_active = np.array(h)[np.array(active)]
        if len(h_active) > 1:
            stats.h_covs.append(float(h_active.std() / max(h_active.mean(), 1e-300)))
        else:
            stats.h_covs.append(0.0)

        h_eff = jnp.clip(
            jnp.where(active, jnp.minimum(h, tf - t), h),
            h_min, h_max
        )

        if use_implicit:
            tn = t + h_eff
            yn, ok, ni = _newton_be(rhs, problem.jac, y, tn, h_eff, params,
                                    tol=newton_tol, max_iter=newton_max_iter)
            n_nit = n_nit + ni
            # simple error estimate: norm of Newton residual proxy
            f_old = jax.vmap(rhs, in_axes=(0, 0, 0))(t, y, params)
            f_new = jax.vmap(rhs, in_axes=(0, 0, 0))(tn, yn, params)
            err_est = _err_norm(y, yn, h_eff[:, None] * (f_new - f_old) / 2., rtol, atol)
            acc = ok & active & (err_est <= 1.)
            t   = jnp.where(acc, tn, t)
            y   = jnp.where(acc[:, None], yn, y)
            se  = jnp.where(err_est > 0, err_est, 1e-300)
            fac = jnp.clip(safety * se**(-0.5), min_factor, max_factor)
            fac_rej = jnp.clip(safety * se**(-0.5), min_factor, 1.0)
            h   = jnp.where(active,
                            jnp.clip(jnp.where(acc, h_eff*fac, h_eff*fac_rej), h_min, h_max),
                            h)
            n_acc = n_acc + acc.astype(jnp.int32)
            n_rej = n_rej + (~acc & active).astype(jnp.int32)
        else:
            step_fn = lambda t_, y_, h_, p_: _rk45_step(rhs, t_, y_, h_, p_)
            yc, ev  = jax.vmap(step_fn)(t, y, h_eff, params)
            err     = _err_norm(y, yc, ev, rtol, atol)
            acc     = (err <= 1.) & active
            t       = jnp.where(acc, t + h_eff, t)
            y       = jnp.where(acc[:, None], yc, y)
            se      = jnp.where(err > 0, err, 1e-300)
            fa      = jnp.where(err == 0., max_factor,
                                jnp.clip(safety*se**(-0.2), min_factor, max_factor))
            fr      = jnp.clip(safety*se**(-0.2), min_factor, 1.)
            h       = jnp.where(active,
                                jnp.clip(jnp.where(acc, h_eff*fa, h_eff*fr), h_min, h_max),
                                h)
            n_acc = n_acc + acc.astype(jnp.int32)
            n_rej = n_rej + (~acc & active).astype(jnp.int32)

        done = done | (t >= tf - 1e-12)
        stats.n_steps_total += 1

    stats.wall_time  = time.perf_counter() - t_start
    stats.n_accept   = np.array(n_acc)
    stats.n_reject   = np.array(n_rej)
    stats.n_newton   = np.array(n_nit)
    return np.array(y), stats

print('Baseline solver defined')

Baseline solver defined


In [30]:
# ---------------------------------------------------------------------------
# Strategy 2: COMPACTION — repack only active trajectories each iteration
# ---------------------------------------------------------------------------
def solve_compaction_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    t0, tf = problem.t_span
    rhs = problem.rhs
    use_implicit = (problem.solver_hint == 'implicit') and (problem.jac is not None)

    B          = len(y0_batch)
    y_full     = np.array(y0_batch, dtype=np.float64)
    p_full     = np.array(params_batch, dtype=np.float64)
    t_full     = np.full(B, float(t0))
    h_full     = np.full(B, h_init)
    done_full  = np.zeros(B, dtype=bool)
    n_acc_full = np.zeros(B, dtype=np.int32)
    n_rej_full = np.zeros(B, dtype=np.int32)
    n_nit_full = np.zeros(B, dtype=np.int32)
    # index mapping: active_indices[i] = original trajectory index
    active_idx = np.arange(B)

    stats = SolverStats()
    t_start = time.perf_counter()
    t_compact = 0.0

    # Working arrays (compacted)
    y  = jnp.array(y_full)
    p  = jnp.array(p_full)
    t  = jnp.array(t_full)
    h  = jnp.array(h_full)
    n_acc = jnp.zeros(B, jnp.int32)
    n_rej = jnp.zeros(B, jnp.int32)
    n_nit = jnp.zeros(B, jnp.int32)

    for step in range(max_iters):
        Bc = int(y.shape[0])
        if Bc == 0:
            break

        stats.active_fracs.append(Bc / B)
        h_np = np.array(h)
        stats.h_covs.append(float(h_np.std() / max(h_np.mean(), 1e-300)) if Bc > 1 else 0.)

        h_eff = jnp.clip(jnp.minimum(h, tf - t), h_min, h_max)

        if use_implicit:
            tn  = t + h_eff
            yn, ok, ni = _newton_be(rhs, problem.jac, y, tn, h_eff, p,
                                    tol=newton_tol, max_iter=newton_max_iter)
            n_nit = n_nit + ni
            f_new = jax.vmap(rhs, in_axes=(0, 0, 0))(tn, yn, p)
            f_old = jax.vmap(rhs, in_axes=(0, 0, 0))(t,  y,  p)
            ee    = _err_norm(y, yn, h_eff[:, None]*(f_new-f_old)/2., rtol, atol)
            acc   = ok & (ee <= 1.)
            t     = jnp.where(acc, tn, t)
            y     = jnp.where(acc[:, None], yn, y)
            se    = jnp.where(ee > 0, ee, 1e-300)
            fac     = jnp.clip(safety*se**(-0.5), min_factor, max_factor)
            fac_rej = jnp.clip(safety*se**(-0.5), min_factor, 1.0)
            h     = jnp.clip(jnp.where(acc, h_eff*fac, h_eff*fac_rej), h_min, h_max)
        else:
            yc, ev = jax.vmap(lambda t_,y_,h_,p_: _rk45_step(rhs,t_,y_,h_,p_))(t,y,h_eff,p)
            err    = _err_norm(y, yc, ev, rtol, atol)
            acc    = err <= 1.
            t      = jnp.where(acc, t + h_eff, t)
            y      = jnp.where(acc[:, None], yc, y)
            se     = jnp.where(err > 0, err, 1e-300)
            fa     = jnp.where(err == 0., max_factor,
                               jnp.clip(safety*se**(-0.2), min_factor, max_factor))
            fr     = jnp.clip(safety*se**(-0.2), min_factor, 1.)
            h      = jnp.clip(jnp.where(acc, h_eff*fa, h_eff*fr), h_min, h_max)

        n_acc = n_acc + acc.astype(jnp.int32)
        n_rej = n_rej + (~acc).astype(jnp.int32)
        stats.n_steps_total += 1

        # Compaction: remove finished trajectories
        tc0 = time.perf_counter()
        t_np  = np.array(t)
        finished = t_np >= tf - 1e-12
        if np.any(finished):
            fin_orig = active_idx[finished]
            y_full[fin_orig]     = np.array(y)[finished]
            t_full[fin_orig]     = t_np[finished]
            n_acc_full[fin_orig] = np.array(n_acc)[finished]
            n_rej_full[fin_orig] = np.array(n_rej)[finished]
            n_nit_full[fin_orig] = np.array(n_nit)[finished]
            keep = ~finished
            active_idx = active_idx[keep]
            y     = jnp.array(np.array(y)[keep])
            p     = jnp.array(np.array(p)[keep])
            t     = jnp.array(t_np[keep])
            h     = jnp.array(np.array(h)[keep])
            n_acc = jnp.array(np.array(n_acc)[keep])
            n_rej = jnp.array(np.array(n_rej)[keep])
            n_nit = jnp.array(np.array(n_nit)[keep])
        t_compact += time.perf_counter() - tc0

    # Flush remaining
    if len(active_idx) > 0:
        y_full[active_idx]     = np.array(y)
        n_acc_full[active_idx] = np.array(n_acc)
        n_rej_full[active_idx] = np.array(n_rej)
        n_nit_full[active_idx] = np.array(n_nit)

    stats.wall_time           = time.perf_counter() - t_start
    stats.compaction_overhead = t_compact
    stats.n_accept            = n_acc_full
    stats.n_reject            = n_rej_full
    stats.n_newton            = n_nit_full
    return y_full, stats

print('Compaction solver defined')

Compaction solver defined


In [31]:
# ---------------------------------------------------------------------------
# Strategy 3: REBATCH — compact every K steps instead of every step
# ---------------------------------------------------------------------------
def solve_rebatch_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    rebatch_every: int = 50,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    t0, tf = problem.t_span
    rhs = problem.rhs
    use_implicit = (problem.solver_hint == 'implicit') and (problem.jac is not None)

    B           = len(y0_batch)
    y_full      = np.array(y0_batch, dtype=np.float64)
    p_full      = np.array(params_batch, dtype=np.float64)
    done_full   = np.zeros(B, dtype=bool)
    n_acc_full  = np.zeros(B, dtype=np.int32)
    n_rej_full  = np.zeros(B, dtype=np.int32)
    n_nit_full  = np.zeros(B, dtype=np.int32)
    active_idx  = np.arange(B)

    y     = jnp.array(y_full)
    p     = jnp.array(p_full)
    t     = jnp.full(B, float(t0))
    h     = jnp.full(B, h_init)
    done  = jnp.zeros(B, dtype=bool)
    n_acc = jnp.zeros(B, jnp.int32)
    n_rej = jnp.zeros(B, jnp.int32)
    n_nit = jnp.zeros(B, jnp.int32)

    stats   = SolverStats()
    t_start = time.perf_counter()
    t_compact = 0.0

    for step in range(max_iters):
        active = ~done
        n_active = int(jnp.sum(active))
        if n_active == 0:
            break

        stats.active_fracs.append(n_active / B)
        h_act = np.array(h)[np.array(active)]
        stats.h_covs.append(float(h_act.std()/max(h_act.mean(),1e-300)) if len(h_act)>1 else 0.)

        h_eff = jnp.clip(
            jnp.where(active, jnp.minimum(h, tf - t), h),
            h_min, h_max
        )

        if use_implicit:
            tn  = t + h_eff
            yn, ok, ni = _newton_be(rhs, problem.jac, y, tn, h_eff, p,
                                    tol=newton_tol, max_iter=newton_max_iter)
            n_nit = n_nit + ni
            f_new = jax.vmap(rhs, in_axes=(0, 0, 0))(tn, yn, p)
            f_old = jax.vmap(rhs, in_axes=(0, 0, 0))(t,  y,  p)
            ee    = _err_norm(y, yn, h_eff[:, None]*(f_new-f_old)/2., rtol, atol)
            acc   = ok & active & (ee <= 1.)
            t     = jnp.where(acc, tn, t)
            y     = jnp.where(acc[:, None], yn, y)
            se    = jnp.where(ee > 0, ee, 1e-300)
            fac     = jnp.clip(safety*se**(-0.5), min_factor, max_factor)
            fac_rej = jnp.clip(safety*se**(-0.5), min_factor, 1.0)
            h     = jnp.where(active, jnp.clip(jnp.where(acc, h_eff*fac, h_eff*fac_rej), h_min, h_max), h)
        else:
            yc, ev = jax.vmap(lambda t_,y_,h_,p_: _rk45_step(rhs,t_,y_,h_,p_))(t,y,h_eff,p)
            err = _err_norm(y, yc, ev, rtol, atol)
            acc = (err <= 1.) & active
            t   = jnp.where(acc, t + h_eff, t)
            y   = jnp.where(acc[:, None], yc, y)
            se  = jnp.where(err > 0, err, 1e-300)
            fa  = jnp.where(err == 0., max_factor,
                            jnp.clip(safety*se**(-0.2), min_factor, max_factor))
            fr  = jnp.clip(safety*se**(-0.2), min_factor, 1.)
            h   = jnp.where(active, jnp.clip(jnp.where(acc, h_eff*fa, h_eff*fr), h_min, h_max), h)

        n_acc = n_acc + acc.astype(jnp.int32)
        n_rej = n_rej + (~acc & active).astype(jnp.int32)
        done  = done | (t >= tf - 1e-12)
        stats.n_steps_total += 1

        # Periodic compaction
        if (step + 1) % rebatch_every == 0 and bool(jnp.any(done)):
            tc0 = time.perf_counter()
            done_np = np.array(done)
            fin_orig = active_idx[done_np]
            y_full[fin_orig]     = np.array(y)[done_np]
            n_acc_full[fin_orig] = np.array(n_acc)[done_np]
            n_rej_full[fin_orig] = np.array(n_rej)[done_np]
            n_nit_full[fin_orig] = np.array(n_nit)[done_np]
            keep = ~done_np
            active_idx = active_idx[keep]
            y     = jnp.array(np.array(y)[keep])
            p     = jnp.array(np.array(p)[keep])
            t     = jnp.array(np.array(t)[keep])
            h     = jnp.array(np.array(h)[keep])
            n_acc = jnp.array(np.array(n_acc)[keep])
            n_rej = jnp.array(np.array(n_rej)[keep])
            n_nit = jnp.array(np.array(n_nit)[keep])
            done  = jnp.zeros(len(active_idx), dtype=bool)
            t_compact += time.perf_counter() - tc0

    if len(active_idx) > 0:
        y_full[active_idx]     = np.array(y)
        n_acc_full[active_idx] = np.array(n_acc)
        n_rej_full[active_idx] = np.array(n_rej)
        n_nit_full[active_idx] = np.array(n_nit)

    stats.wall_time           = time.perf_counter() - t_start
    stats.compaction_overhead = t_compact
    stats.n_accept            = n_acc_full
    stats.n_reject            = n_rej_full
    stats.n_newton            = n_nit_full
    return y_full, stats

print('Rebatch solver defined')

Rebatch solver defined


In [32]:
# ---------------------------------------------------------------------------
# Strategy 4: BUCKETS — partition batch by stiffness regime at t=0,
# then solve each sub-batch independently (avoids step-size spread divergence).
# Buckets determined by spread in estimated stiffness (max |λ| via param proxy).
# ---------------------------------------------------------------------------
def solve_buckets_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    n_buckets: int = 4,
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    B = len(y0_batch)
    t0, tf = problem.t_span

    # Stiffness proxy: max param value per trajectory
    stiff_proxy = np.max(np.abs(params_batch), axis=1)  # (B,)
    bucket_edges = np.percentile(stiff_proxy, np.linspace(0, 100, n_buckets + 1))
    bucket_ids   = np.digitize(stiff_proxy, bucket_edges[1:-1])

    y_full     = np.zeros_like(y0_batch)
    n_acc_full = np.zeros(B, dtype=np.int32)
    n_rej_full = np.zeros(B, dtype=np.int32)
    n_nit_full = np.zeros(B, dtype=np.int32)

    combined_stats = SolverStats()
    t_start = time.perf_counter()

    for bid in range(n_buckets):
        idx = np.where(bucket_ids == bid)[0]
        if len(idx) == 0:
            continue
        yb  = y0_batch[idx]
        pb  = params_batch[idx]
        # Each bucket gets an h_init tuned to its stiffness range
        sp_mean = stiff_proxy[idx].mean()
        h0  = float(np.clip(0.1 / max(sp_mean, 1.), h_min, h_max))
        y_out, st = solve_baseline_jax(
            problem, yb, pb,
            rtol=rtol, atol=atol,
            h_init=h0, h_min=h_min, h_max=h_max,
            newton_tol=newton_tol, newton_max_iter=newton_max_iter,
            max_iters=max_iters, safety=safety,
            min_factor=min_factor, max_factor=max_factor,
        )
        y_full[idx]     = y_out
        n_acc_full[idx] = st.n_accept
        n_rej_full[idx] = st.n_reject
        n_nit_full[idx] = st.n_newton
        combined_stats.n_steps_total  += st.n_steps_total
        combined_stats.active_fracs   += st.active_fracs
        combined_stats.h_covs         += st.h_covs

    combined_stats.wall_time = time.perf_counter() - t_start
    combined_stats.n_accept  = n_acc_full
    combined_stats.n_reject  = n_rej_full
    combined_stats.n_newton  = n_nit_full
    return y_full, combined_stats

print('Bucket solver defined')

Bucket solver defined


In [33]:
# ---------------------------------------------------------------------------
# Strategy 5: SPLIT QUEUE — explicit queue for nonstiff/mild, implicit for stiff.
# Classifies by stiffness proxy threshold at start.
# ---------------------------------------------------------------------------
def solve_split_queue_jax(
    problem: ODEProblem,
    y0_batch, params_batch,
    stiffness_threshold: float = 50.0,   # max |param| above this → implicit
    rtol=1e-6, atol=1e-8,
    h_init=1e-3, h_min=1e-8, h_max=0.5,
    newton_tol=1e-10, newton_max_iter=20,
    max_iters=500_000,
    safety=0.9, min_factor=0.2, max_factor=5.0,
) -> Tuple[np.ndarray, SolverStats]:
    B = len(y0_batch)
    stiff_proxy = np.max(np.abs(params_batch), axis=1)
    impl_mask   = stiff_proxy >= stiffness_threshold
    expl_mask   = ~impl_mask

    y_full     = np.zeros_like(y0_batch)
    n_acc_full = np.zeros(B, dtype=np.int32)
    n_rej_full = np.zeros(B, dtype=np.int32)
    n_nit_full = np.zeros(B, dtype=np.int32)

    combined = SolverStats()
    t_start = time.perf_counter()

    # Explicit sub-batch: override solver hint
    if np.any(expl_mask):
        ei = np.where(expl_mask)[0]
        import copy
        expl_problem = copy.copy(problem)
        expl_problem.solver_hint = 'explicit'
        yo, se = solve_compaction_jax(
            expl_problem, y0_batch[ei], params_batch[ei],
            rtol=rtol, atol=atol, h_init=h_init, h_min=h_min, h_max=h_max,
            newton_tol=newton_tol, newton_max_iter=newton_max_iter,
            max_iters=max_iters, safety=safety,
            min_factor=min_factor, max_factor=max_factor,
        )
        y_full[ei]     = yo
        n_acc_full[ei] = se.n_accept
        n_rej_full[ei] = se.n_reject
        n_nit_full[ei] = se.n_newton
        combined.n_steps_total += se.n_steps_total
        combined.active_fracs  += se.active_fracs
        combined.h_covs        += se.h_covs
        print(f'  Explicit queue: {len(ei)} trajectories, {se.n_steps_total} steps')

    # Implicit sub-batch
    if np.any(impl_mask) and problem.jac is not None:
        ii = np.where(impl_mask)[0]
        import copy
        impl_problem = copy.copy(problem)
        impl_problem.solver_hint = 'implicit'
        h0_impl = float(np.clip(0.1 / max(stiff_proxy[impl_mask].mean(), 1.), h_min, h_max))
        yo, si = solve_compaction_jax(
            impl_problem, y0_batch[ii], params_batch[ii],
            rtol=rtol, atol=atol, h_init=h0_impl, h_min=h_min, h_max=h_max,
            newton_tol=newton_tol, newton_max_iter=newton_max_iter,
            max_iters=max_iters, safety=safety,
            min_factor=min_factor, max_factor=max_factor,
        )
        y_full[ii]     = yo
        n_acc_full[ii] = si.n_accept
        n_rej_full[ii] = si.n_reject
        n_nit_full[ii] = si.n_newton
        combined.n_steps_total += si.n_steps_total
        combined.active_fracs  += si.active_fracs
        combined.h_covs        += si.h_covs
        print(f'  Implicit queue: {len(ii)} trajectories, {si.n_steps_total} steps')

    combined.wall_time = time.perf_counter() - t_start
    combined.n_accept  = n_acc_full
    combined.n_reject  = n_rej_full
    combined.n_newton  = n_nit_full
    return y_full, combined

print('Split-queue solver defined')

Split-queue solver defined


## 5. Correctness Verification

In [34]:
# ---------------------------------------------------------------------------
# Quick correctness check on problems with known reference solutions
# ---------------------------------------------------------------------------
VERIFY_BATCH = 32
rng_v = np.random.default_rng(42)

print('Correctness check (baseline solver, batch=32):')
print(f'{"Problem":>8}  {"MaxErr":>12}  {"Wall(s)":>9}  {"MeanAcc":>9}')
print('-' * 50)

for name, prob in PROBLEMS.items():
    if prob.ref_solution is None:
        continue
    y0  = prob.ic_generator(rng_v, VERIFY_BATCH)
    par = prob.param_sampler(rng_v, VERIFY_BATCH)
    tf  = prob.t_span[1]

    h0 = 1e-4 if prob.stiffness_label in ('strong', 'extreme') else 1e-3
    y_out, st = solve_baseline_jax(
        prob, y0, par,
        rtol=1e-6, atol=1e-8, h_init=h0,
        max_iters=1_000_000,
    )
    y_ref = prob.ref_solution(tf, y0, par)
    max_err = float(np.max(np.abs(y_out - y_ref)))
    print(f'{name:>8}  {max_err:>12.3e}  {st.wall_time:>9.3f}  {st.n_accept.mean():>9.1f}')

Correctness check (baseline solver, batch=32):
 Problem        MaxErr    Wall(s)    MeanAcc
--------------------------------------------------
      NS     4.826e-05      8.215      146.5
      MS     5.640e-07      2.994       92.3


TypeError: sub got incompatible shapes for broadcasting: (4,), (32,).

## 6. Strategy Comparison Benchmark

In [ ]:
# ---------------------------------------------------------------------------
# Benchmark setup — shared across all per-problem cells below
# ---------------------------------------------------------------------------
BENCH_PROBLEMS = ['MS', 'SS', 'MX', 'VZ', 'HE', 'AD']
BATCH_SIZE     = 256

# Resume: load any previously completed results from Drive
_ckpt = load_from_drive(CKPT_MAIN)
all_results = _ckpt if _ckpt is not None else {}
if all_results:
    print(f'[resume] Loaded {len(all_results)} completed result(s) from Drive.')

STRATEGIES = {
    'baseline':   lambda p, y0, par: solve_baseline_jax(p, y0, par, max_iters=500_000),
    'compaction': lambda p, y0, par: solve_compaction_jax(p, y0, par, max_iters=500_000),
    'rebatch_50': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=50, max_iters=500_000),
    'rebatch_10': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=10, max_iters=500_000),
    'buckets_4':  lambda p, y0, par: solve_buckets_jax(p, y0, par, n_buckets=4, max_iters=500_000),
    'split_q':    lambda p, y0, par: solve_split_queue_jax(p, y0, par, max_iters=500_000),
}

def _run_problem_bench(pname, batch_size=BATCH_SIZE, seed=None, strategy_overrides=None):
    """Run all strategies on one problem; skip completed pairs; save after each."""
    prob = PROBLEMS[pname]
    rng  = np.random.default_rng(seed if seed is not None else hash(pname) % (2**31))
    y0   = prob.ic_generator(rng, batch_size)
    par  = prob.param_sampler(rng, batch_size)
    strats = strategy_overrides if strategy_overrides is not None else STRATEGIES
    print(f'\n=== {pname} ({prob.stiffness_label}, dim={prob.state_dim}, '
          f't∈{prob.t_span}, B={batch_size}) ===')
    for sname, solver_fn in strats.items():
        key = (pname, sname)
        if key in all_results:
            s = all_results[key].summary()
            print(f'  {sname:>14}: [resumed] wall={s["wall_time"]:.3f}s')
            continue
        try:
            y_out, st = solver_fn(prob, y0, par)
            all_results[key] = st
            save_to_drive(all_results, CKPT_MAIN)
            s = st.summary()
            print(f'  {sname:>14}: wall={s["wall_time"]:.3f}s  '
                  f'active={s["mean_active_frac"]:.1%}  '
                  f'h_cov={s["mean_h_cov"]:.3f}  rej={s["rej_rate"]:.1%}  '
                  f'newton={s["mean_newton"]:.1f}  compact={s["compaction_overhead"]:.3f}s')
        except Exception as e:
            print(f'  {sname:>14}: FAILED — {e}')

# ----- MS: Mildly stiff (λ∈[1,10], t_end=6) -----
_run_problem_bench('MS', seed=1337)


In [ ]:
# ----- SS: Strongly stiff (λ∈[100,2000], t_end=1) -----
# Explicit RK45 must use tiny steps (h~1/2000); implicit BE trivially stable.
_run_problem_bench('SS', seed=2001)


In [ ]:
# ----- MX: Mixed stiffness (fast λ∈[500,1000] + slow λ∈[0.5,5], ε=0.01, t_end=2) -----
# Tests h-spread between fast/slow modes within the same trajectory.
_run_problem_bench('MX', seed=3001)


In [ ]:
# ----- VZ: Van der Pol (μ∈[1,50], t_end=10) -----
# Stiffness O(μ²)~2500. Use B=128 and max_iters=200_000 to bound runtime.
_vz_overrides = {
    'baseline':   lambda p, y0, par: solve_baseline_jax(p, y0, par, max_iters=200_000),
    'compaction': lambda p, y0, par: solve_compaction_jax(p, y0, par, max_iters=200_000),
    'rebatch_50': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=50, max_iters=200_000),
    'rebatch_10': lambda p, y0, par: solve_rebatch_jax(p, y0, par, rebatch_every=10, max_iters=200_000),
    'buckets_4':  lambda p, y0, par: solve_buckets_jax(p, y0, par, n_buckets=4, max_iters=200_000),
    'split_q':    lambda p, y0, par: solve_split_queue_jax(p, y0, par, max_iters=200_000),
}
_run_problem_bench('VZ', batch_size=128, seed=4001, strategy_overrides=_vz_overrides)


In [ ]:
# ----- HE: 1D Heat equation MoL (PDE parabolic, d=20, α∈[0.1,2], t_end=0.5) -----
# Tridiagonal 20×20 Jacobian — exercises batched jnp.linalg.solve on GPU.
# Step-size spread scales with α; trajectories with large α need tiny h.
_run_problem_bench('HE', seed=5001)


In [ ]:
# ----- AD: 1D Advection MoL (PDE hyperbolic, d=20, c∈[0.2,2], t_end=1) -----
# Explicit only (nonstiff). CFL h_stable ∝ 1/c → 10× step-size spread.
# Best case for COMPACTION and BUCKETS: done trajectories (high c) finish early.
_run_problem_bench('AD', seed=6001)


## 7. Large Batch Scaling

In [ ]:
# ---------------------------------------------------------------------------
# Scaling study: baseline vs compaction on SS — small batches [64, 128, 256]
# ---------------------------------------------------------------------------
SCALE_PROBLEM = 'SS'
prob_s  = PROBLEMS[SCALE_PROBLEM]
rng_s   = np.random.default_rng(999)

# Resume: load existing scale records
_scale_ckpt = load_from_drive(CKPT_SCALE)
scale_records = _scale_ckpt if _scale_ckpt is not None else []
_done_bs = {r['bs'] for r in scale_records}
if _done_bs:
    print(f'[resume] Loaded scale records for batches: {sorted(_done_bs)}')

print(f'Scaling study on {SCALE_PROBLEM} — small batches:')
print(f'{"Batch":>6} {"Baseline(s)":>13} {"Compact(s)":>12} {"Speedup":>9} '
      f'{"ActiveFrac":>12} {"hCoV":>8}')
print('-'*65)

for bs in [64, 128, 256]:
    if bs in _done_bs:
        r = next(r for r in scale_records if r['bs'] == bs)
        print(f'{bs:>6} {r["t_base"]:>13.3f} {r["t_comp"]:>12.3f} '
              f'{r["speedup"]:>9.2f}x {r["af_comp"]:>12.1%} '
              f'{r["hcov_comp"]:>8.3f}  [resumed]')
        continue
    y0  = prob_s.ic_generator(rng_s, bs)
    par = prob_s.param_sampler(rng_s, bs)
    _, st_b = solve_baseline_jax(prob_s, y0, par, max_iters=200_000)
    _, st_c = solve_compaction_jax(prob_s, y0, par, max_iters=200_000)
    sb = st_b.summary(); sc = st_c.summary()
    speedup = sb['wall_time'] / max(sc['wall_time'], 1e-9)
    rec = dict(bs=bs, t_base=sb['wall_time'], t_comp=sc['wall_time'], speedup=speedup,
               af_base=sb['mean_active_frac'], af_comp=sc['mean_active_frac'],
               hcov_base=sb['mean_h_cov'],    hcov_comp=sc['mean_h_cov'])
    scale_records.append(rec)
    save_to_drive(scale_records, CKPT_SCALE)
    _done_bs.add(bs)
    print(f'{bs:>6} {sb["wall_time"]:>13.3f} {sc["wall_time"]:>12.3f} '
          f'{speedup:>9.2f}x {sc["mean_active_frac"]:>12.1%} '
          f'{sc["mean_h_cov"]:>8.3f}')


In [ ]:
# ---------------------------------------------------------------------------
# Scaling study continued — large batches [512, 1024]
# (split to avoid >30 min per cell on CPU; GPU runs these quickly)
# ---------------------------------------------------------------------------
rng_sl = np.random.default_rng(9999)
_done_bs_large = {r['bs'] for r in scale_records}

print(f'Scaling study on {SCALE_PROBLEM} — large batches:')
print(f'{"Batch":>6} {"Baseline(s)":>13} {"Compact(s)":>12} {"Speedup":>9} '
      f'{"ActiveFrac":>12} {"hCoV":>8}')
print('-'*65)

for bs in [512, 1024]:
    if bs in _done_bs_large:
        r = next(r for r in scale_records if r['bs'] == bs)
        print(f'{bs:>6} {r["t_base"]:>13.3f} {r["t_comp"]:>12.3f} '
              f'{r["speedup"]:>9.2f}x {r["af_comp"]:>12.1%} '
              f'{r["hcov_comp"]:>8.3f}  [resumed]')
        continue
    y0  = prob_s.ic_generator(rng_sl, bs)
    par = prob_s.param_sampler(rng_sl, bs)
    _, st_b = solve_baseline_jax(prob_s, y0, par, max_iters=200_000)
    _, st_c = solve_compaction_jax(prob_s, y0, par, max_iters=200_000)
    sb = st_b.summary(); sc = st_c.summary()
    speedup = sb['wall_time'] / max(sc['wall_time'], 1e-9)
    rec = dict(bs=bs, t_base=sb['wall_time'], t_comp=sc['wall_time'], speedup=speedup,
               af_base=sb['mean_active_frac'], af_comp=sc['mean_active_frac'],
               hcov_base=sb['mean_h_cov'],    hcov_comp=sc['mean_h_cov'])
    scale_records.append(rec)
    save_to_drive(scale_records, CKPT_SCALE)
    _done_bs_large.add(bs)
    print(f'{bs:>6} {sb["wall_time"]:>13.3f} {sc["wall_time"]:>12.3f} '
          f'{speedup:>9.2f}x {sc["mean_active_frac"]:>12.1%} '
          f'{sc["mean_h_cov"]:>8.3f}')


## 8. Large-Dimension Test (LD)

In [ ]:
# ---------------------------------------------------------------------------
# Large-dim (d=64) benchmark: GPU register pressure + batched linear solve
# ---------------------------------------------------------------------------
prob_ld = PROBLEMS['LD']
LD_BATCHES = [16, 32, 64, 128]
rng_ld = np.random.default_rng(7777)

# Resume: load existing LD records
_ld_ckpt  = load_from_drive(CKPT_LD)
ld_records = _ld_ckpt if _ld_ckpt is not None else {}
if ld_records:
    print(f'[resume] Loaded LD records for batches: {sorted(ld_records.keys())}')

print(f'Large-dimension benchmark (d={prob_ld.state_dim}):')
print(f'{"Batch":>6} {"Baseline(s)":>13} {"Compact(s)":>12} '
      f'{"MaxErr(base)":>14} {"MaxErr(comp)":>14}')
print('-'*65)

for bs in LD_BATCHES:
    if bs in ld_records:
        r = ld_records[bs]
        print(f'{bs:>6} {r["t_base"]:>13.3f} {r["t_comp"]:>12.3f} '
              f'{r["err_b"]:>14.3e} {r["err_c"]:>14.3e}  [resumed]')
        continue
    y0  = prob_ld.ic_generator(rng_ld, bs)
    par = prob_ld.param_sampler(rng_ld, bs)
    tf  = prob_ld.t_span[1]

    y_b, st_b = solve_baseline_jax(prob_ld, y0, par, h_init=1e-4, max_iters=200_000)
    y_c, st_c = solve_compaction_jax(prob_ld, y0, par, h_init=1e-4, max_iters=200_000)
    y_ref = prob_ld.ref_solution(tf, y0, par)
    err_b = float(np.max(np.abs(y_b - y_ref)))
    err_c = float(np.max(np.abs(y_c - y_ref)))
    ld_records[bs] = dict(t_base=st_b.wall_time, t_comp=st_c.wall_time,
                          err_b=err_b, err_c=err_c)
    save_to_drive(ld_records, CKPT_LD)
    print(f'{bs:>6} {st_b.wall_time:>13.3f} {st_c.wall_time:>12.3f} '
          f'{err_b:>14.3e} {err_c:>14.3e}')


## 9. Visualization

In [ ]:
# ---------------------------------------------------------------------------
# Plot 1: Active fraction and h-CoV over time per strategy (SS problem)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
pname_ref = 'SS' if 'SS' in BENCH_PROBLEMS else BENCH_PROBLEMS[0]
fig.suptitle(f'GPU Divergence Instrumentation — {pname_ref} problem, B={BATCH_SIZE}', fontsize=13)

plot_strategies = ['baseline', 'compaction', 'rebatch_50', 'buckets_4']
colors = ['steelblue', 'darkorange', 'forestgreen', 'crimson']

ax_af, ax_hc, ax_rej, ax_nwt = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

for sname, col in zip(plot_strategies, colors):
    key = (pname_ref, sname)
    if key not in all_results:
        continue
    st = all_results[key]
    s  = st.summary()
    label = f'{sname} ({s["wall_time"]:.2f}s)'

    af = np.array(st.active_fracs)
    hc = np.array(st.h_covs)
    xs = np.arange(len(af))

    ax_af.plot(xs, af, color=col, alpha=0.7, label=label, lw=1)
    ax_hc.plot(xs, hc, color=col, alpha=0.7, label=label, lw=1)

ax_af.set_xlabel('Iteration'); ax_af.set_ylabel('Active fraction')
ax_af.set_title('Active Fraction over Time'); ax_af.legend(fontsize=7); ax_af.grid(True, alpha=0.3)

ax_hc.set_xlabel('Iteration'); ax_hc.set_ylabel('Step-size CoV')
ax_hc.set_title('Step-size Spread (CoV) over Time'); ax_hc.legend(fontsize=7); ax_hc.grid(True, alpha=0.3)

# Bar: rejection rate per (problem, strategy)
strategy_names = list(STRATEGIES.keys())
prob_names     = BENCH_PROBLEMS
x = np.arange(len(strategy_names))
width = 0.2
for i, pn in enumerate(prob_names):
    rr = [all_results.get((pn,sn), SolverStats(n_accept=np.zeros(1),n_reject=np.zeros(1),n_newton=np.zeros(1))).summary()['rej_rate']
          if (pn,sn) in all_results else 0.
          for sn in strategy_names]
    ax_rej.bar(x + i*width, rr, width, label=pn, alpha=0.8)
ax_rej.set_xticks(x + width*(len(prob_names)-1)/2)
ax_rej.set_xticklabels(strategy_names, rotation=20, ha='right', fontsize=8)
ax_rej.set_ylabel('Rejection rate'); ax_rej.set_title('Step Rejection Rate by Strategy & Problem')
ax_rej.legend(fontsize=8); ax_rej.grid(True, alpha=0.3, axis='y')

# Bar: mean Newton iters/step
for i, pn in enumerate(prob_names):
    nw = [all_results.get((pn,sn), SolverStats(n_accept=np.zeros(1),n_reject=np.zeros(1),n_newton=np.zeros(1))).summary()['mean_newton']
          if (pn,sn) in all_results else 0.
          for sn in strategy_names]
    ax_nwt.bar(x + i*width, nw, width, label=pn, alpha=0.8)
ax_nwt.set_xticks(x + width*(len(prob_names)-1)/2)
ax_nwt.set_xticklabels(strategy_names, rotation=20, ha='right', fontsize=8)
ax_nwt.set_ylabel('Mean Newton iters/traj'); ax_nwt.set_title('Newton Iterations by Strategy & Problem')
ax_nwt.legend(fontsize=8); ax_nwt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
_fig1_local = 'gpu_bench_divergence.png'
plt.savefig(_fig1_local, dpi=150, bbox_inches='tight')
plt.show()
print('Saved gpu_bench_divergence.png')
if IN_COLAB:
    import shutil
    shutil.copy(_fig1_local, os.path.join(DRIVE_DIR, _fig1_local))
    print('  → also saved to Drive')

In [ ]:
# ---------------------------------------------------------------------------
# Plot 2: Wall time comparison and scaling
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('GPU Strategy Comparison', fontsize=13)

# 2a: Wall time per strategy per problem
ax = axes[0]
x  = np.arange(len(BENCH_PROBLEMS))
w  = 0.10
for i, sname in enumerate(STRATEGIES.keys()):
    wts = [all_results[(pn,sname)].summary()['wall_time']
           if (pn,sname) in all_results else 0.
           for pn in BENCH_PROBLEMS]
    ax.bar(x + i*w, wts, w, label=sname, alpha=0.85)
ax.set_xticks(x + w*(len(STRATEGIES)-1)/2)
ax.set_xticklabels(BENCH_PROBLEMS)
ax.set_ylabel('Wall time (s)'); ax.set_title('Wall Time by Strategy')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis='y')

# 2b: Scaling — baseline vs compaction
ax = axes[1]
bs_arr = np.array([r['bs'] for r in scale_records])
t_base = np.array([r['t_base'] for r in scale_records])
t_comp = np.array([r['t_comp'] for r in scale_records])
ax.loglog(bs_arr, t_base, '-o', label='baseline',   color='steelblue')
ax.loglog(bs_arr, t_comp, '-s', label='compaction', color='darkorange')
ax.set_xlabel('Batch size'); ax.set_ylabel('Wall time (s)')
ax.set_title(f'Scaling on {SCALE_PROBLEM} (baseline vs compaction)')
ax.legend(); ax.grid(True, which='both', alpha=0.3)

# 2c: Compaction overhead vs speedup
ax = axes[2]
speedups = np.array([r['speedup'] for r in scale_records])
ax.semilogx(bs_arr, speedups, '-^', color='forestgreen', lw=2, ms=8)
ax.axhline(1., ls='--', color='gray', lw=1, label='break-even')
ax.set_xlabel('Batch size'); ax.set_ylabel('Baseline / Compaction')
ax.set_title('Compaction Speedup over Baseline')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
_fig2_local = 'gpu_bench_strategies.png'
plt.savefig(_fig2_local, dpi=150, bbox_inches='tight')
plt.show()
print('Saved gpu_bench_strategies.png')
if IN_COLAB:
    import shutil
    shutil.copy(_fig2_local, os.path.join(DRIVE_DIR, _fig2_local))
    print('  → also saved to Drive')

In [ ]:
# ---------------------------------------------------------------------------
# Plot 3: Per-problem active fraction (all BENCH_PROBLEMS, baseline strategy)
# ---------------------------------------------------------------------------
n_bench = len(BENCH_PROBLEMS)
n_cols  = 3
n_rows  = (n_bench + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.5 * n_rows))
fig.suptitle('Active Fraction Decay — Baseline Strategy, B=' + str(BATCH_SIZE), fontsize=12)

for idx, pname in enumerate(BENCH_PROBLEMS):
    ax  = axes.flat[idx]
    key = (pname, 'baseline')
    if key not in all_results:
        ax.set_visible(False); continue
    st = all_results[key]
    af = np.array(st.active_fracs)
    ax.fill_between(np.arange(len(af)), af, alpha=0.4, color='steelblue')
    ax.plot(np.arange(len(af)), af, lw=1, color='steelblue')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Iteration'); ax.set_ylabel('Active fraction')
    prob = PROBLEMS[pname]
    ax.set_title(f'{pname} ({prob.stiffness_label})')
    ax.grid(True, alpha=0.3)
    wasted = 1.0 - np.mean(af)
    ax.text(0.98, 0.05, f'Mean wasted: {wasted:.1%}',
            transform=ax.transAxes, ha='right', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

for idx in range(n_bench, n_rows * n_cols):
    axes.flat[idx].set_visible(False)

plt.tight_layout()
_fig3_local = 'gpu_bench_active_frac.png'
plt.savefig(_fig3_local, dpi=150, bbox_inches='tight')
plt.show()
print('Saved gpu_bench_active_frac.png')
if IN_COLAB:
    import shutil
    shutil.copy(_fig3_local, os.path.join(DRIVE_DIR, _fig3_local))
    print('  → also saved to Drive')


In [ ]:
# ---------------------------------------------------------------------------
# Final summary table
# ---------------------------------------------------------------------------
print('\n' + '='*100)
print(f'{"Problem":>8} {"Strategy":>15} {"Wall(s)":>9} {"ActiveFrac":>11} '
      f'{"hCoV":>7} {"RejRate":>9} {"Newton/t":>10} {"CompactOH":>11}')
print('='*100)
for pname in BENCH_PROBLEMS:
    for sname in STRATEGIES:
        key = (pname, sname)
        if key not in all_results:
            continue
        s = all_results[key].summary()
        print(f'{pname:>8} {sname:>15} {s["wall_time"]:>9.3f} '
              f'{s["mean_active_frac"]:>11.2%} '
              f'{s["mean_h_cov"]:>7.3f} '
              f'{s["rej_rate"]:>9.2%} '
              f'{s["mean_newton"]:>10.2f} '
              f'{s["compaction_overhead"]:>11.3f}')
    print('-'*100)
print('='*100)
# Save summary CSV to Drive
if IN_COLAB:
    import csv
    _csv_path = os.path.join(DRIVE_DIR, 'bench_summary.csv')
    _fields = ['problem', 'strategy', 'wall_time', 'mean_active_frac',
               'mean_h_cov', 'rej_rate', 'mean_newton', 'compaction_overhead']
    with open(_csv_path, 'w', newline='') as _f:
        _w = csv.DictWriter(_f, fieldnames=_fields)
        _w.writeheader()
        for pname in BENCH_PROBLEMS:
            for sname in STRATEGIES:
                key = (pname, sname)
                if key not in all_results:
                    continue
                s = all_results[key].summary()
                _w.writerow(dict(problem=pname, strategy=sname,
                                 wall_time=round(s['wall_time'], 4),
                                 mean_active_frac=round(s['mean_active_frac'], 6),
                                 mean_h_cov=round(s['mean_h_cov'], 6),
                                 rej_rate=round(s['rej_rate'], 6),
                                 mean_newton=round(s['mean_newton'], 4),
                                 compaction_overhead=round(s['compaction_overhead'], 4)))
    print(f'Summary CSV saved \u2192 {_csv_path}')
